# Tutorial 09 — Ремоделирование семейств волокон

Notebook сопоставляет прямую ротацию, эволюцию непрерывного распределения, структурное интегрирование и когортный turnover. Все примеры синтетические и предназначены для верификации.

## 1. Окружение и импорты

In [ ]:
LANGUAGE = "ru"
from pathlib import Path
import sys


def _find_repository_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Repository root not found. Open the notebook inside the repository.")


REPOSITORY_ROOT = _find_repository_root(Path.cwd().resolve())
SOURCE_DIRECTORY = REPOSITORY_ROOT / "src"
if str(SOURCE_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIRECTORY))

import biomechanics_tutorials
print("Repository:", REPOSITORY_ROOT)
print("Package:", Path(biomechanics_tutorials.__file__).resolve())


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from biomechanics_tutorials.fiber_family_remodeling import (
    FiberFamily,
    FiberMaterialParameters,
    ODFRemodelingParameters,
    ReorientationParameters,
    TurnoverParameters,
    angular_integrated_energy,
    axial_angle_difference,
    axial_grid,
    axial_von_mises_density,
    discrete_family_energy,
    planar_structure_tensor_energy,
    simulate_cohort_turnover,
    simulate_discrete_reorientation,
    simulate_odf_remodeling,
)
from biomechanics_tutorials.plotting import apply_tutorial_style

apply_tutorial_style()


## 2. Прямая переориентация существующего семейства
Директор изменяется непрерывно, но синтез и удаление отсутствуют.

In [ ]:
time = np.linspace(0.0, 8.0, 401)
target_angle = np.deg2rad(30.0)
direct = simulate_discrete_reorientation(
    time,
    initial_angle=np.deg2rad(-50.0),
    target=target_angle,
    parameters=ReorientationParameters(rate=0.8),
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(time, np.rad2deg(direct["angle"]), label="fiber family")
ax.plot(time, np.rad2deg(direct["target"]), "--", label="mechanical cue")
ax.set_xlabel("time")
ax.set_ylabel("axial angle, deg")
ax.legend();


## 3. Непрерывное распределение ориентаций
Дрейф выравнивает плотность, а вращательная диффузия расширяет её. Вероятность должна сохранять нормировку и неотрицательность.

In [ ]:
theta = axial_grid(181)
initial_density = axial_von_mises_density(theta, np.deg2rad(-45.0), 5.0)
odf_time = np.linspace(0.0, 6.0, 241)
odf = simulate_odf_remodeling(
    odf_time,
    theta,
    initial_density,
    np.deg2rad(35.0),
    ODFRemodelingParameters(alignment_rate=1.0, rotational_diffusivity=0.015),
)

fig, ax = plt.subplots(figsize=(8, 4))
for index in [0, 40, 100, 240]:
    ax.plot(np.rad2deg(theta), odf["density"][index], label=f"t={odf_time[index]:.1f}")
ax.set_xlabel("axial orientation, deg")
ax.set_ylabel("orientation density")
ax.legend();


## 4. Интегрирование Ланира, дискретные семейства и структурное замыкание
Это разные редукции структурной идеи, которые не обязаны совпадать для широких и многомодальных распределений.

In [ ]:
stretches = np.linspace(1.0, 1.22, 61)
theta_fine = axial_grid(721)
density = axial_von_mises_density(theta_fine, np.deg2rad(20.0), 6.0)
material = FiberMaterialParameters(k1=2.0, k2=6.0)

angular = np.array([
    angular_integrated_energy(np.diag([lam, 1 / np.sqrt(lam)]), theta_fine, density, material)
    for lam in stretches
])
closure = np.array([
    planar_structure_tensor_energy(np.diag([lam, 1 / np.sqrt(lam)]), np.deg2rad(20.0), 0.16, material)
    for lam in stretches
])
discrete = np.array([
    discrete_family_energy(
        np.diag([lam, 1 / np.sqrt(lam)]),
        [FiberFamily(np.deg2rad(20.0))],
        material,
    )
    for lam in stretches
])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(stretches, angular, label="angular integration")
ax.plot(stretches, closure, "--", label="structure-tensor closure")
ax.plot(stretches, discrete, ":", label="single discrete family")
ax.set_xlabel("stretch")
ax.set_ylabel("fiber energy")
ax.legend();


## 5. Когортный turnover
Существующие когорты сохраняют направления отложения. Ансамбль перестраивается через выживание и замещение.

In [ ]:
turnover_time = np.linspace(0.0, 24.0, 241)
turnover = simulate_cohort_turnover(
    turnover_time,
    stress_protocol=lambda t: 1.0 if t < 4.0 else 1.35,
    deposition_angle_protocol=lambda t: np.deg2rad(-20.0) if t < 6.0 else np.deg2rad(55.0),
    parameters=TurnoverParameters(
        half_life=5.0,
        basal_production=0.18,
        stress_gain=0.8,
        target_stress=1.0,
        deposition_stretch=1.08,
    ),
    initial_angle=np.deg2rad(-20.0),
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(turnover_time, np.rad2deg(turnover["mean_angle"]))
axes[0].set_xlabel("time")
axes[0].set_ylabel("mean axial angle, deg")
axes[1].plot(turnover_time, turnover["mass"], label="mass")
axes[1].plot(turnover_time, turnover["mean_age"], label="mean cohort age")
axes[1].set_xlabel("time")
axes[1].legend();


## 6. Итоги верификации

In [ ]:
final_direct_error = np.rad2deg(abs(direct["error"][-1]))
final_odf_error = np.rad2deg(
    abs(axial_angle_difference(np.deg2rad(35.0), odf["mean_angle"][-1]))
)
print(f"Direct-family final error: {final_direct_error:.3f} deg")
print(f"ODF final mean-angle error: {final_odf_error:.3f} deg")
print(f"Turnover final mass: {turnover['mass'][-1]:.3f}")
print(f"Turnover final mean cohort age: {turnover['mean_age'][-1]:.3f}")


## Исследовательская интерпретация
Сходный конечный угол не доказывает сходный биологический механизм. Нужно сравнивать скрытые переменные, массу, возраст, предрастяжение и прогнозы при новом нагружении.